# G28 — Training RoleNet with masked listener features: adaptation and cost (5-fold CV, train+val, 10 seeds; test untouched)

**Question.** If the listener's (L) face features are missing, does training RoleNet with the *same* kind of masking
improve the forecast under that masking, and what does it cost on the data as they are? The architecture is unchanged;
only the training policy differs. This is not a test of a new architecture.

**Arms** (RoleNet `Full` of G13/G14, same folds, inner early-stop episodes and hyper-parameters; 10 seeds, same seed
per fold for both arms):
* `R-std` — RoleNet trained as before.
* `R-mask` — **training matched to the evaluation masking**: for each training MCIS, with probability 0.5 one of the
  conditions C1–C3 that *changes that MCIS* is drawn uniformly and applied; if none applies, the MCIS is left as is.
  The actual masked share and the frequency of each condition are reported (the share is below 50% because some MCIS
  have no L). A policy that does not help does not show that other policies cannot.

**Masking conditions** (applied after role assignment, to the L face-token branch only):

| Condition | What is removed | MCIS on which it is evaluated |
|---|---|---|
| C0 | nothing (data as they are) | all, and for each Ck also on exactly the Ck set |
| C1 | all L face observations (clips I–III) | at least one L observation |
| C2 | L in clip III, L history (I/II) kept | L in III **and** L in I or II |
| C3 | L history (I/II), L in clip III kept | L in III **and** L in I or II |

* The mask is applied to the frame mask **before** the face tokens are built, so the masked cells become the learned
  *absent* token exactly like naturally missing cells, and the face auxiliary head sees the same masked tokens as the
  main head.
* Speech tokens, scene tokens (whole-frame and face CLIP features) and voice cues (which include per-role mouth–audio
  synchrony) are **kept**. The intervention is therefore "missing role-specific face features of L", not "L was never
  observed".
* C1 and C3 give **presence patterns** that the role rule can also produce naturally (L exists only when clip III has at
  least two identities, so L absent in III implies L absent everywhere). The masked MCIS are still artificial: e.g. after
  C1 the voice cues may still carry L information, which a natural MCIS without L does not have. C2 creates a pattern
  the role rule never produces (L in I/II without L in III).
* Modality dropout (whole face branch / whole context branch, as in G8b) is kept identical in both arms. The share of
  L-masked training MCIS that also had the whole face branch dropped (so the L mask had no effect on the main head) is
  reported.

**Checkpoint selection (differs from G13).** Both arms select the epoch by the same inner-dev criterion
J_dev = 0.5·NLL(C0) + 0.5·mean_k NLL(Ck), each NLL(Ck) on the dev MCIS eligible for Ck (G13 used dev UAR on C0).
`R-std` keeps its training, only the selection rule is G28's.

**Estimand and interval (fixed before running).** NLL is the only primary metric.
* NLL_{s,k}: mean NLL of seed s's probabilities on the Ck set under Ck.
* **Δ_rec = (1/3S) Σ_k Σ_s [NLL^std_{s,k} − NLL^mask_{s,k}]** — mean of per-seed NLLs, not the NLL of the ensemble
  (ensemble NLL is reported separately). Δ_rec > 0 means `R-mask` forecasts better under masking.
* Interval: 2,000 bootstrap draws resampling the 10 seeds and the 45 episodes, with the **same** draws for both arms and
  all three conditions; every quantity is recomputed per draw (C1–C3 are not treated as independent).
* Secondary: Δ_clean = NLL^mask(C0) − NLL^std(C0) on all MCIS (> 0 = cost of masked training on the data as they are);
  sensitivity_k = NLL^std(Ck) − NLL^std(C0) on the same Ck set; per-condition Δ_rec,k; UAR (descriptive).
* Recovery share Δ_rec,k / sensitivity_k is reported only when the sensitivity's CI lower bound is > 0; it is not a share
  of "recovered information".

**Reading rules (fixed before running).**

| Result | Allowed conclusion |
|---|---|
| CI of Δ_rec entirely > 0 | masked training helps adaptation to the interventions tried |
| CI of Δ_rec contains 0 | not shown that this policy helps; **cannot tell** whether the drop under masking is mostly lost evidence or lack of adaptation |
| CI of Δ_rec entirely < 0 | this policy makes the forecast under masking worse |

A positive Δ_rec together with a Δ_clean whose CI is above 0 is a trade-off, not robustness without cost.

**Auxiliary analysis (not a gate).** Do the 9 presence flags (A/L/O × I/II/III, after role assignment) forecast B's
emotion better than the training-fold prior? Regularised logistic regression (C chosen inside the training fold by
grouped inner CV) and the same with pairwise flag interactions; out-of-fold by episode; ΔNLL vs prior with an episode
bootstrap. A gain only shows an association of the presence pattern with the label, not that it adds information
beyond the observed content.

In [ ]:
# ======== CONFIG ========
import os


def first_existing(*paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"none of {paths}")


DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
FEATURES_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2"
SPLIT_CSV = first_existing("/kaggle/input/datasets/ptrnghieu/hi-ef-split/source_folder_split_seed42.csv",
                           "/kaggle/input/hi-ef-split/source_folder_split_seed42.csv")
G8A_DIR = first_existing("/kaggle/input/datasets/ptrnghieu/g8a-features", "/kaggle/input/g8a-features")
OUT_DIR = "/kaggle/working"

N_OUTER, N_INNER_DEV = 5, 5
SEEDS = [42, 123, 456, 7, 11, 19, 23, 31, 37, 43]    # as G14
LR, WEIGHT_DECAY = 1e-4, 1e-5                # B1 settings (only needed by the shared model cell)
FC_EPOCHS, PATIENCE, FC_BATCH = 50, 8, 32
RN = dict(D=128, heads=4, layers=2, dropout=0.2, lr=3e-4, wd=1e-2, epochs=80, patience=12, batch=64,
          aux_w=0.3, a_w=0.3, p_drop_ctx=0.3, p_drop_face=0.15)     # G8b, unchanged
PCA_DIM, MAXF, MAXF_POOL = 128, 24, 32
SAME_PERSON_COS, DOMINANT_MIN_FRAC = 0.45, 0.25
VOICE_SAME_COS = 0.35
DEBUG_PER_EPISODE = None     # e.g. 6 for a quick smoke test

FULL = dict(role=True, faces=True, ctx=True, aux=True, mdrop=True)
BASE = dict(readout='query', nq=1, q_readonly=False, role_emb=True, clip_emb=True, absent='token',
            drop=(), speech_parts=('text', 'audio', 'voice'), clips=(0, 1, 2), seed_offset=0)
ARMS = ['R-std', 'R-mask']
EXPERIMENTS = [("RoleNet", 'role', FULL)]    # only used by the shared model cell's parameter print
P_MASK, N_BOOT = 0.5, 2000
LR_GRID = [0.01, 0.1, 1.0, 10.0]             # auxiliary presence analysis

In [ ]:
import os, json, math, random, time
import numpy as np
import pandas as pd

EMO = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
POL = ['positive', 'neutral', 'negative']
E2I = {e: i for i, e in enumerate(EMO)}
P2I = {p: i for i, p in enumerate(POL)}

# Reference numbers from the locked-split report (validation, 5-seed mean)
REPORT_REF = {'B1_full': (24.65, 35.79), 'T1_future_KL': (25.34, 35.65),
              'Frozen A recognizer (E_A)': (23.21, 33.41)}


def load_tables(annot_csv, split_csv):
    """annotation.csv has no header: 0 clip_id, 1 text, 5 polarity, 6 intensity, 7 emotion, 8 uncertainty."""
    ann = pd.read_csv(annot_csv, header=None, dtype=str).set_index(0)
    sp = pd.read_csv(split_csv, dtype=str)

    def text(c):
        t = ann.at[c, 1] if c in ann.index else None
        return t if isinstance(t, str) else ''

    for k in (1, 2, 3):
        sp[f't{k}'] = sp[f'clip{k}'].map(text)
    sp['yA'] = sp['clip3_emotion'].map(E2I)
    sp['yB'] = sp['clip4_emotion'].map(E2I)
    sp['pA'] = sp['clip3'].map(lambda c: P2I.get(ann.at[c, 5], -1))
    assert sp[['yA', 'yB']].notna().all().all(), 'missing A/B emotion labels'
    return ann, sp


def eval_rows(sp, split, unlock_test=False):
    if split == 'test' and not unlock_test:
        raise RuntimeError('Test split is locked. Set UNLOCK_TEST = True only for the final, preregistered run.')
    return sp[sp['split'] == split].reset_index(drop=True)


def war_uar(pred, y, k):
    pred, y = np.asarray(pred), np.asarray(y)
    war = (pred == y).mean() * 100
    uar = np.mean([(pred[y == c] == c).mean() * 100 for c in range(k) if (y == c).any()])
    return war, uar


def source_boot_ci(pred, y, src, k, n_boot=2000, seed=0):
    """95% CI by resampling whole source folders (episodes) with replacement."""
    pred, y, src = np.asarray(pred), np.asarray(y), np.asarray(src)
    rng = np.random.default_rng(seed)
    groups = [np.where(src == s)[0] for s in np.unique(src)]
    stats = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        stats.append(war_uar(pred[idx], y[idx], k))
    lo, hi = np.percentile(np.array(stats), [2.5, 97.5], axis=0)
    return lo, hi


def report(name, pred, y, src, k=7):
    war, uar = war_uar(pred, y, k)
    lo, hi = source_boot_ci(pred, y, src, k)
    print(f'{name:<46} UAR {uar:5.2f} [{lo[1]:5.1f},{hi[1]:5.1f}]   WAR {war:5.2f} [{lo[0]:5.1f},{hi[0]:5.1f}]')
    return {'name': name, 'UAR': uar, 'WAR': war, 'UAR_lo': lo[1], 'UAR_hi': hi[1], 'WAR_lo': lo[0], 'WAR_hi': hi[0]}


def transition_tables(train_rows, alpha=1.0):
    """P(B | E_A) and P(B | E_A, P_A) estimated on TRAIN gold pairs, add-alpha smoothing."""
    T = np.full((7, 7), alpha)
    TP = np.full((7, 3, 7), alpha)
    for a, p, b in zip(train_rows['yA'], train_rows['pA'], train_rows['yB']):
        T[a, b] += 1
        if p >= 0:
            TP[a, p, b] += 1
    return T / T.sum(1, keepdims=True), TP / TP.sum(2, keepdims=True)


def rtt_forecast(pA_emo, T, pA_pol=None, TP=None):
    """Recognize-then-Transition: B distribution from A posteriors.
    Returns hard (argmax of transition row of argmax A) and soft (expected) B predictions."""
    hard = T[pA_emo.argmax(1)].argmax(1)
    if pA_pol is not None and TP is not None:
        pB = np.einsum('na,np,apb->nb', pA_emo, pA_pol, TP)  # assumes E_A and P_A posteriors independent
    else:
        pB = pA_emo @ T
    return hard, pB.argmax(1), pB

import glob, pickle
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ANNOT_CSV = glob.glob(os.path.join(DATASET_DIR, "*", "Hi-EF", "annotation.csv"))[0]
ann, sp = load_tables(ANNOT_CSV, SPLIT_CSV)
sp = sp[sp.split.isin(['train', 'val'])].reset_index(drop=True)      # test rows are dropped here
assert 'test' not in set(sp.split)
if DEBUG_PER_EPISODE:
    sp = sp.groupby('source_folder', group_keys=False).head(DEBUG_PER_EPISODE).reset_index(drop=True)
DEV = sp
train_all = ev = DEV          # names used by the shared feature-loading cell
N = len(DEV)
EPS = np.array(sorted(DEV.source_folder.unique()))
print(f"development MCIS {N} | episodes {len(EPS)}")


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# ---- load every clip used by any MCIS (I-IV) once, keep it on the GPU
all_clips = sorted(set(sp[['clip1', 'clip2', 'clip3', 'clip4']].values.ravel()) & set(
    f[:-3].replace('_', '/', 1) for f in os.listdir(FEATURES_DIR) if f.endswith('.pt')))
CIDX = {c: i for i, c in enumerate(all_clips)}
missing = [c for c in set(train_all[['clip1', 'clip2', 'clip3']].values.ravel()) | set(ev[['clip1', 'clip2', 'clip3']].values.ravel())
           if c not in CIDX]
assert not missing, f"{len(missing)} clips without features, e.g. {missing[:3]}"

bufs = {k: [] for k in ('face', 'fmask', 'ori', 'text', 'audio', 'afound')}
for c in tqdm(all_clips, desc='loading features'):
    d = torch.load(os.path.join(FEATURES_DIR, c.replace('/', '_') + '.pt'), map_location='cpu', weights_only=False)
    face = d['face_features'].float()
    fm = d.get('face_valid_mask')
    bufs['face'].append(face)
    bufs['fmask'].append(torch.ones(face.shape[0], dtype=torch.bool) if fm is None else torch.as_tensor(fm).bool().reshape(-1))
    bufs['ori'].append(d['ori_features'].float())
    bufs['text'].append(d['text_feature'].float().reshape(-1))
    bufs['audio'].append(d.get('audio_feature', torch.zeros(527)).float().reshape(-1))
    bufs['afound'].append(torch.tensor(bool(d.get('audio_found', True))))
FEAT = {k: torch.stack(v).to(DEVICE) for k, v in bufs.items()}
del bufs
print({k: tuple(v.shape) for k, v in FEAT.items()})


def gather(idx):
    """idx: LongTensor of clip indices (any shape) -> dict of feature tensors with that leading shape."""
    flat = idx.reshape(-1)
    return {k: v[flat].reshape(*idx.shape, *v.shape[1:]) for k, v in FEAT.items()}

In [ ]:
class TemporalEncoder(nn.Module):
    def __init__(self, d=512, n_frames=16, layers=2, heads=8, dropout=0.1):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, n_frames, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)

    def forward(self, x, mask):  # mask: True = valid frame
        mask = mask.clone()
        mask[~mask.any(1), 0] = True
        h = self.enc(x + self.pos[:, :x.size(1)], src_key_padding_mask=~mask)
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1)


class ClipEncoder(nn.Module):
    """Face/original temporal encoders + text/audio tokens -> 1-layer fusion Transformer -> one 512-d vector."""

    def __init__(self, d=512):
        super().__init__()
        self.face, self.ori = TemporalEncoder(d), TemporalEncoder(d)
        self.text = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, d))
        self.audio = nn.Sequential(nn.LayerNorm(527), nn.Linear(527, d))
        self.modality = nn.Parameter(torch.randn(1, 4, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 8, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.fusion = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)

    def forward(self, b):
        ori_mask = torch.ones(b['ori'].shape[:2], dtype=torch.bool, device=b['ori'].device)
        tokens = torch.stack([self.face(b['face'], b['fmask']), self.ori(b['ori'], ori_mask),
                              self.text(b['text']), self.audio(F.normalize(b['audio'], dim=-1))], 1)
        valid = torch.ones(tokens.shape[:2], dtype=torch.bool, device=tokens.device)
        valid[:, 3] = b['afound']
        h = self.fusion(tokens + self.modality, src_key_padding_mask=~valid)
        m = valid.unsqueeze(-1).float()
        return self.norm((h * m).sum(1) / m.sum(1))


class ClipRecognizer(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.enc = ClipEncoder(d)
        self.drop = nn.Dropout(0.3)
        self.emo, self.pol = nn.Linear(d, 7), nn.Linear(d, 3)

    def forward(self, b):
        h = self.drop(self.enc(b))
        return self.emo(h), self.pol(h)


N_REC = 12   # 7 emotion probs + 3 polarity probs + max prob + entropy


class Forecaster(nn.Module):
    def __init__(self, use_raw=True, use_traj=False, d=512, positions=None):
        super().__init__()
        self.use_raw, self.use_traj = use_raw, use_traj
        self.positions = positions   # clip positions (0=I, 1=II, 2=III); None = the last n clips
        self.enc = ClipEncoder(d) if use_raw else None
        self.traj = nn.Sequential(nn.LayerNorm(N_REC), nn.Linear(N_REC, d), nn.GELU(), nn.Linear(d, d)) if use_traj else None
        self.clip_pos = nn.Parameter(torch.randn(1, 3, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 8, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.inter = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, d // 2), nn.GELU(),
                                  nn.Dropout(0.2), nn.Linear(d // 2, 7))

    def forward(self, clip_idx, rec):  # clip_idx [B,n], rec [B,n,N_REC], n <= 3 clips in temporal order
        B, n = clip_idx.shape
        tok = 0
        if self.use_raw:
            feats = gather(clip_idx)
            flat = {k: v.reshape(B * n, *v.shape[2:]) for k, v in feats.items()}
            tok = self.enc(flat).reshape(B, n, -1)
        if self.use_traj:
            tok = tok + self.traj(rec)
        pos = self.clip_pos[:, list(self.positions)] if self.positions is not None else self.clip_pos[:, 3 - n:]
        h = self.inter(tok + pos)
        return self.head(h.mean(1))

## From G8a features to role-tagged slots

In [ ]:
from collections import defaultdict
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA

G8 = {}
for f in sorted(glob.glob(os.path.join(G8A_DIR, '**', 'shard_*.pkl'), recursive=True)):
    G8.update(pickle.load(open(f, 'rb')))
need = sorted(set(DEV[['clip1', 'clip2', 'clip3']].values.ravel()))
miss = [c for c in need if c not in G8]
assert not miss, f"{len(miss)} clips missing from G8a, e.g. {miss[:3]}"


def softmax(z):
    e = np.exp(z - z.max(-1, keepdims=True))
    return e / e.sum(-1, keepdims=True)


# per-clip frame features except the PCA part (18 dims), and the raw HSEmotion embeddings
FB, EMB = {}, {}
for c in need:
    fs = G8[c]['faces']
    if not fs:
        FB[c], EMB[c] = np.zeros((0, 18), np.float32), None
        continue
    dur = max(G8[c]['meta'].get('duration') or 0.0, 1e-3)
    fer = np.stack([d['fer'] for d in fs]).astype(np.float32)
    box = np.stack([d['box'] for d in fs])
    FB[c] = np.concatenate([
        softmax(fer[:, :8]), fer[:, 8:10], np.stack([d['pose'] for d in fs]) / 90.0,
        np.stack([(box[:, 0] + box[:, 2]) / 2, (box[:, 1] + box[:, 3]) / 2,
                  np.sqrt(np.clip((box[:, 2] - box[:, 0]) * (box[:, 3] - box[:, 1]), 0, None))], 1),
        np.nan_to_num(np.array([[d['mouth'] * 10] for d in fs], np.float32)),
        np.array([[min(d['t'] / dur, 1.0)] for d in fs], np.float32)], 1).astype(np.float32)
    EMB[c] = np.stack([d['fer_emb'] for d in fs]) if all('fer_emb' in d for d in fs) else None
HAS_EMB = all(EMB[c] is not None for c in need if len(G8[c]['faces']))
FDIM = (PCA_DIM if HAS_EMB else 0) + 18
print(f"HSEmotion embedding available: {HAS_EMB} | frame feature dim {FDIM}")


def pick_frames(idx, cap):
    return idx if len(idx) <= cap else [idx[i] for i in np.linspace(0, len(idx) - 1, cap).astype(int)]


def sync(c, face_ids):
    # correlation of mouth opening with the audio energy envelope, and mouth variability, for a set of faces
    A = G8[c]['audio']
    fs = [G8[c]['faces'][j] for j in face_ids]
    fs = [d for d in fs if np.isfinite(d['mouth'])]
    if A is None or len(fs) < 4:
        return 0.0, 0.0
    m = np.array([d['mouth'] for d in fs])
    env = A['env']
    e = np.array([env[min(int(d['t'] * 10), len(env) - 1)] for d in fs]) if len(env) else np.zeros(len(fs))
    r = float(np.corrcoef(m, e)[0, 1]) if m.std() > 1e-6 and e.std() > 1e-9 else 0.0
    return r, float(m.std() * 10)


def mean12(c_faces, n_sampled):
    if not c_faces:
        return np.zeros(12, np.float32)
    fer = np.stack([d['fer'] for d in c_faces]).astype(np.float32)
    v = np.concatenate([softmax(fer[:, :8]), fer[:, 8:10]], 1).mean(0)
    return np.concatenate([v, [1.0, len({d['frame'] for d in c_faces}) / max(n_sampled, 1)]]).astype(np.float32)


NVOICE = 3 * 2 + 3
SLOT, PSLOT = {}, {}                      # (n, role, clip) / (n, clip) -> (clip id, face indices)
FMASK = np.zeros((N, 3, 3, MAXF), bool); PMASK = np.zeros((N, 1, 3, MAXF_POOL), bool)
VOI = np.zeros((N, 3, NVOICE), np.float32)
LRF = np.zeros((N, 60), np.float32)       # G6b-style role means (late fusion and the joint-branch arm)
CENT_COS = np.full(N, np.nan, np.float32)
for n, row in enumerate(tqdm(DEV.itertuples(), total=N, desc='roles')):
    cl = [row.clip1, row.clip2, row.clip3]
    items = [(k, j) for k, c in enumerate(cl) for j in range(len(G8[c]['faces']))]
    lab = np.zeros(len(items), int)
    E = np.stack([G8[cl[k]]['faces'][j]['arc'] for k, j in items]).astype(np.float32) if items else None
    if len(items) > 1:
        lab = AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average',
                                      distance_threshold=1 - SAME_PERSON_COS).fit_predict(E)
    frames = defaultdict(set)
    for (k, j), p in zip(items, lab):
        frames[(k, p)].add(G8[cl[k]]['faces'][j]['frame'])
    ids3 = sorted({p for (k, p) in frames if k == 2}, key=lambda p: -len(frames[(2, p)]))
    A = ids3[0] if ids3 else None
    L = ids3[1] if len(ids3) > 1 else None
    if A is not None and L is not None:
        ca, cb = E[lab == A].mean(0), E[lab == L].mean(0)
        CENT_COS[n] = float(ca @ cb / (np.linalg.norm(ca) * np.linalg.norm(cb) + 1e-9))
    role = lambda p: 0 if p == A else (1 if p == L else 2)
    by = defaultdict(list)
    for (k, j), p in zip(items, lab):
        by[(role(p), k)].append(j)
        by[('pool', k)].append(j)
    for k, c in enumerate(cl):
        for r in range(3):
            idx = pick_frames(sorted(by[(r, k)], key=lambda j: G8[c]['faces'][j]['t']), MAXF)
            SLOT[(n, r, k)] = (c, idx); FMASK[n, r, k, :len(idx)] = True
        idx = pick_frames(sorted(by[('pool', k)], key=lambda j: G8[c]['faces'][j]['t']), MAXF_POOL)
        PSLOT[(n, k)] = (c, idx); PMASK[n, 0, k, :len(idx)] = True
        v = [x for r in range(3) for x in sync(c, by[(r, k)])]
        a3, ak = G8[cl[2]]['audio'], G8[c]['audio']
        ok = a3 is not None and ak is not None and a3.get('ecapa') is not None and ak.get('ecapa') is not None
        vcos = float(a3['ecapa'].astype(np.float32) @ ak['ecapa'].astype(np.float32)) if ok else 0.0
        VOI[n, k] = v + [vcos, float(ok), len(set(lab)) / 5.0]

    def faces_of(k, p):
        return [G8[cl[k]]['faces'][j] for (kk, j), q in zip(items, lab) if kk == k and q == p]

    def dominant(k):
        ns = G8[cl[k]]['meta']['n_sampled']
        cand = sorted({q for (kk, q) in frames if kk == k}, key=lambda q: -len(frames[(k, q)]))
        return cand[0] if cand and len(frames[(k, cand[0])]) / max(ns, 1) >= DOMINANT_MIN_FRAC else None

    n3 = G8[cl[2]]['meta']['n_sampled']
    blocks = [mean12(faces_of(2, A) if A is not None else [], n3), mean12(faces_of(2, L) if L is not None else [], n3),
              mean12((faces_of(0, L) + faces_of(1, L)) if L is not None else [], n3)]
    for k in (1, 0):
        d = dominant(k)
        blocks.append(mean12(faces_of(k, d) if d is not None else [], G8[cl[k]]['meta']['n_sampled']))
    LRF[n] = np.concatenate(blocks)

VIS = FMASK[:, 1, 2].any(-1)
print(f"A found in III {FMASK[:, 0, 2].any(-1).mean() * 100:.1f}% | listener visible in III {VIS.mean() * 100:.1f}% | "
      f"listener also in I/II {FMASK[:, 1, :2].any((-1, -2)).mean() * 100:.1f}%")


def build_face_tensors(fit_clips):
    # PCA of the HSEmotion embedding fitted on faces of the training clips of the current fold only
    pca, var = None, None
    if HAS_EMB:
        pool = np.concatenate([EMB[c] for c in fit_clips if EMB.get(c) is not None])
        pick = np.random.default_rng(0).choice(len(pool), min(60000, len(pool)), replace=False)
        pca = PCA(PCA_DIM, random_state=0).fit(pool[pick].astype(np.float32))
        var = float(pca.explained_variance_ratio_.sum())
    FV = {}
    for c in need:
        if len(FB[c]) == 0:
            FV[c] = np.zeros((0, FDIM), np.float32)
        else:
            FV[c] = np.concatenate([pca.transform(EMB[c].astype(np.float32)), FB[c]], 1) if HAS_EMB else FB[c]
    Fa = np.zeros((N, 3, 3, MAXF, FDIM), np.float16)
    Pa = np.zeros((N, 1, 3, MAXF_POOL, FDIM), np.float16)
    for (n, r, k), (c, idx) in SLOT.items():
        if idx:
            Fa[n, r, k, :len(idx)] = FV[c][idx]
    for (n, k), (c, idx) in PSLOT.items():
        if idx:
            Pa[n, 0, k, :len(idx)] = FV[c][idx]
    return torch.tensor(Fa, device=DEVICE), torch.tensor(Pa, device=DEVICE), var

## Models

In [ ]:
T = lambda a, dt=None: torch.tensor(a, device=DEVICE) if dt is None else torch.tensor(a, dtype=dt, device=DEVICE)
FMASK, PMASK, VOI = T(FMASK), T(PMASK), T(VOI)
FACE = POOL = LRFZ = None             # set per fold
CLIPIDX = T([[CIDX[c] for c in r] for r in DEV[['clip1', 'clip2', 'clip3']].values])
TXT, AUD, AFD = FEAT['text'][CLIPIDX], F.normalize(FEAT['audio'][CLIPIDX], dim=-1), FEAT['afound'][CLIPIDX].float()
fm = FEAT['fmask'][CLIPIDX].unsqueeze(-1).float()
SCN = torch.cat([FEAT['ori'][CLIPIDX].mean(2), (FEAT['face'][CLIPIDX] * fm).sum(2) / fm.sum(2).clamp(min=1)], -1)
ZREC = torch.zeros(N, 3, N_REC, device=DEVICE)
YB, YA = T(DEV.yB.values), T(DEV.yA.values)


class FramePool(nn.Module):
    def __init__(self, fin, d):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(fin), nn.Linear(fin, d), nn.GELU(), nn.Linear(d, d))
        self.score = nn.Linear(d, 1)

    def forward(self, x, m):                      # x [..., F, fin], m [..., F]
        h = self.proj(x.float())
        a = self.score(h).squeeze(-1).masked_fill(~m, -1e4)
        w = torch.softmax(a, -1) * m.float()
        return (w.unsqueeze(-1) * h).sum(-2), m.any(-1)


class RoleNet(nn.Module):
    def __init__(self, cfg, d=RN['D']):
        super().__init__()
        self.cfg, self.R = cfg, (3 if cfg['role'] else 1)
        if cfg['faces']:
            self.pool = FramePool(FDIM, d)
            self.absent = nn.Parameter(torch.randn(self.R, 3, d) * 0.02)
            self.face_role = nn.Parameter(torch.randn(self.R, d) * 0.02)
        if cfg['ctx']:
            self.text = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, d))
            self.audio = nn.Sequential(nn.LayerNorm(527), nn.Linear(527, d))
            self.voice = nn.Linear(NVOICE, d)
            self.scene = nn.Sequential(nn.LayerNorm(1024), nn.Linear(1024, d))
            self.ctx_role = nn.Parameter(torch.randn(2, d) * 0.02)
        self.clip_emb = nn.Parameter(torch.randn(3, d) * 0.02)
        self.query = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, RN['heads'], 4 * d, RN['dropout'], batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, RN['layers'], enable_nested_tensor=False)
        mk = lambda: nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, 7))
        self.head = mk()
        self.head_face = mk() if cfg['aux'] and cfg['faces'] else None
        self.head_ctx = mk() if cfg['aux'] and cfg['ctx'] else None
        self.head_A = mk() if cfg['aux'] and cfg['faces'] and cfg['role'] else None

    def forward(self, ix, train=False):
        B, groups, aux = len(ix), [], {}
        if self.cfg['faces']:
            x, m = (FACE[ix], FMASK[ix]) if self.R == 3 else (POOL[ix], PMASK[ix])
            h, present = self.pool(x, m)                                      # [B, R, 3, d]
            h = torch.where(present.unsqueeze(-1), h, self.absent.unsqueeze(0).expand(B, -1, -1, -1))
            h = h + self.face_role[None, :, None] + self.clip_emb[None, None]
            ft = h.reshape(B, self.R * 3, -1)
            groups.append(ft)
            if self.head_face is not None:
                aux['face'] = (self.head_face(ft.mean(1)), YB[ix], RN['aux_w'])
            if self.head_A is not None:
                tA = torch.where(present[:, 0, 2], YA[ix], torch.full_like(YA[ix], -100))
                aux['A'] = (self.head_A(h[:, 0, 2]), tA, RN['a_w'])
        if self.cfg['ctx']:
            spk_ = self.text(TXT[ix]) + self.audio(AUD[ix]) * AFD[ix].unsqueeze(-1) + self.voice(VOI[ix]) + self.ctx_role[0]
            scn = self.scene(SCN[ix]) + self.ctx_role[1]
            ct = torch.cat([spk_ + self.clip_emb, scn + self.clip_emb], 1)     # [B, 6, d]
            groups.append(ct)
            if self.head_ctx is not None:
                aux['ctx'] = (self.head_ctx(ct.mean(1)), YB[ix], RN['aux_w'])
        toks = torch.cat([self.query.expand(B, -1, -1)] + groups, 1)
        valid = torch.ones(toks.shape[:2], dtype=torch.bool, device=toks.device)
        if train and self.cfg['mdrop'] and len(groups) == 2:
            u = torch.rand(B, device=toks.device)
            drop_ctx = u < RN['p_drop_ctx']
            drop_face = (u >= RN['p_drop_ctx']) & (u < RN['p_drop_ctx'] + RN['p_drop_face'])
            nf = groups[0].shape[1]
            valid[:, 1:1 + nf] &= ~drop_face.unsqueeze(1)
            valid[:, 1 + nf:] &= ~drop_ctx.unsqueeze(1)
        out = self.enc(toks, src_key_padding_mask=~valid)
        return self.head(out[:, 0]), aux


class B1Wrap(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = Forecaster(use_raw=True, use_traj=False)

    def forward(self, ix, train=False):
        return self.f(CLIPIDX[ix], ZREC[ix]), {}


class B1FaceWrap(nn.Module):
    # B1 + the 60-d G6b-style role-face vector through a zero-initialised branch, trained jointly (the G7b design)
    def __init__(self, n_face=60, d=512):
        super().__init__()
        self.f = Forecaster(use_raw=True, use_traj=False, d=d)
        self.face = nn.Sequential(nn.Linear(n_face, d // 2), nn.GELU(), nn.Dropout(0.3), nn.Linear(d // 2, d))
        nn.init.zeros_(self.face[-1].weight); nn.init.zeros_(self.face[-1].bias)

    def forward(self, ix, train=False):
        b, clip_idx = self.f, CLIPIDX[ix]
        B, n = clip_idx.shape
        feats = gather(clip_idx)
        tok = b.enc({k: v.reshape(B * n, *v.shape[2:]) for k, v in feats.items()}).reshape(B, n, -1)
        h = b.inter(tok + b.clip_pos[:, 3 - n:])
        return b.head(h.mean(1) + self.face(LRFZ[ix])), {}


HP = {'b1': dict(lr=LR, wd=WEIGHT_DECAY, epochs=FC_EPOCHS, patience=PATIENCE, batch=FC_BATCH),
      'role': dict(lr=RN['lr'], wd=RN['wd'], epochs=RN['epochs'], patience=RN['patience'], batch=RN['batch'])}
HP['b1face'] = HP['b1']
MAKE = {'b1': lambda cfg: B1Wrap(), 'b1face': lambda cfg: B1FaceWrap(), 'role': lambda cfg: RoleNet(cfg)}
print("parameters:", {n: f"{sum(p.numel() for p in MAKE[k](c).parameters()) / 1e6:.2f}M" for n, k, c in EXPERIMENTS})


def predict(model, ix, bs=256):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(ix), bs):
            out.append(F.softmax(model(ix[i:i + bs])[0], -1).cpu())
    return torch.cat(out).numpy()


def train_eval(kind, cfg, tr, dev, te, seed):
    seed_all(seed)
    hp = HP[kind]
    model = MAKE[kind](cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=hp['lr'], weight_decay=hp['wd'])
    y_dev = YB[dev].cpu().numpy()
    best, best_state, bad = -1, None, 0
    for ep in range(hp['epochs']):
        model.train()
        perm = tr[torch.randperm(len(tr), device=DEVICE)]
        for i in range(0, len(perm), hp['batch']):
            j = perm[i:i + hp['batch']]
            logits, aux = model(j, train=True)
            loss = F.cross_entropy(logits, YB[j])
            for l, t, w in aux.values():
                if (t >= 0).any():
                    loss = loss + w * F.cross_entropy(l, t, ignore_index=-100)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        u = war_uar(predict(model, dev).argmax(1), y_dev, 7)[1]
        if u > best:
            best, bad = u, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= hp['patience']:
                break
    model.load_state_dict(best_state)
    return predict(model, te), best

## Presence patterns and the MCIS sets of C1–C3

In [ ]:
PRES = FMASK.any(-1).cpu().numpy()                    # [N, role A/L/O, clip I/II/III]
L_any, L3, L12 = PRES[:, 1].any(-1), PRES[:, 1, 2], PRES[:, 1, :2].any(-1)
ELIG = np.stack([L_any, L3 & L12, L3 & L12], 1)      # C1, C2, C3
COND = ['C1 no L', 'C2 no L-III (history kept)', 'C3 no L history (L-III kept)']
for k in range(3):
    print(f"{COND[k]:<32} eligible MCIS {ELIG[:, k].sum():5d} ({ELIG[:, k].mean() * 100:.1f}%)")
print("role-rule checks (expected 0): L in I/II without L in III", int((L12 & ~L3).sum()),
      "| O in III without L in III", int((PRES[:, 2, 2] & ~L3).sum()))
pat = pd.Series([' '.join(r + ':' + ''.join(str(int(x)) for x in PRES[i, j]) for j, r in enumerate('ALO'))
                 for i in range(N)])
vc = pat.value_counts()
print(f"\n{len(vc)} distinct presence patterns (A/L/O, clips I II III); most frequent:")
print((vc.head(15) / N * 100).round(1).to_string())
ELIG_T = T(ELIG)

## Auxiliary analysis: do the presence flags forecast B's emotion? (logistic, out-of-fold)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import PolynomialFeatures

sizes = DEV.source_folder.value_counts()
order = list(sizes.index)
random.Random(0).shuffle(order)
order = sorted(order, key=lambda e: -sizes[e])
load_, FOLD = [0] * N_OUTER, {}
for e in order:
    f = int(np.argmin(load_)); FOLD[e] = f; load_[f] += sizes[e]
fold_of_row = DEV.source_folder.map(FOLD).values
y_all = DEV.yB.values
src = DEV.source_folder.values
print("fold sizes (MCIS):", load_)

X9 = PRES.reshape(N, 9).astype(float)
X_int = PolynomialFeatures(2, interaction_only=True, include_bias=False).fit_transform(X9)


def lr_proba(Xtr, ytr, Xte, C):
    m = LogisticRegression(C=C, max_iter=5000).fit(Xtr, ytr)
    out = np.full((len(Xte), 7), 1e-6)
    out[:, m.classes_] = m.predict_proba(Xte)
    return out / out.sum(1, keepdims=True)


def nll_rows(p, y):
    return -np.log(np.clip(p[np.arange(len(y)), y], 1e-7, None))


PRES_OOF = {k: np.zeros((N, 7)) for k in ('prior', 'flags', 'flags+pairs')}
CHOSEN = {'flags': [], 'flags+pairs': []}
for f in range(N_OUTER):
    tr, te = fold_of_row != f, fold_of_row == f
    cnt = np.bincount(y_all[tr], minlength=7) + 1.0
    PRES_OOF['prior'][te] = cnt / cnt.sum()
    for name, X in (('flags', X9), ('flags+pairs', X_int)):
        gkf = GroupKFold(5)
        score = {}
        for C in LR_GRID:
            s = []
            for a, b in gkf.split(X[tr], y_all[tr], src[tr]):
                s.append(nll_rows(lr_proba(X[tr][a], y_all[tr][a], X[tr][b], C), y_all[tr][b]).mean())
            score[C] = np.mean(s)
        C = min(score, key=score.get)
        CHOSEN[name].append(C)
        PRES_OOF[name][te] = lr_proba(X[tr], y_all[tr], X[te], C)
print("C chosen per fold:", CHOSEN)

NLLP = {k: nll_rows(v, y_all) for k, v in PRES_OOF.items()}
groups = [np.where(src == e)[0] for e in np.unique(src)]
rng = np.random.default_rng(0)
for name in ('flags', 'flags+pairs'):
    d = NLLP[name] - NLLP['prior']
    bs = [d[np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])].mean() for _ in range(N_BOOT)]
    lo, hi = np.percentile(bs, [2.5, 97.5])
    print(f"  {name:<12} NLL {NLLP[name].mean():.4f} vs prior {NLLP['prior'].mean():.4f} | "
          f"ΔNLL (model − prior) {d.mean():+.4f} [{lo:+.4f}, {hi:+.4f}]"
          + ("  -> presence pattern is associated with the label" if hi < 0 else ""))

## RoleNet with L masking applied before the face tokens

In [ ]:
def apply_cond(m, cond):
    # m: [B, 3, 3, F] frame mask (role, clip, frame); cond: [B] in {0 none, 1 C1, 2 C2, 3 C3}
    m = m.clone()
    m[cond == 1, 1] = False                  # C1: L, all clips
    m[cond == 2, 1, 2] = False               # C2: L in clip III
    m[cond == 3, 1, :2] = False              # C3: L in clips I/II
    return m


class RoleNetMask(RoleNet):
    # RoleNet (G8b, all switches on) whose L frame mask can be changed per sample before pooling; the masked cells
    # become the absent token, and the face auxiliary head sees the same tokens as the main head.
    def forward(self, ix, train=False, cond=None):
        B, aux = len(ix), {}
        m = FMASK[ix] if cond is None else apply_cond(FMASK[ix], cond)
        h, present = self.pool(FACE[ix], m)                                   # [B, 3, 3, d]
        h = torch.where(present.unsqueeze(-1), h, self.absent.unsqueeze(0).expand(B, -1, -1, -1))
        h = h + self.face_role[None, :, None] + self.clip_emb[None, None]
        ft = h.reshape(B, 9, -1)
        aux['face'] = (self.head_face(ft.mean(1)), YB[ix], RN['aux_w'])
        tA = torch.where(present[:, 0, 2], YA[ix], torch.full_like(YA[ix], -100))
        aux['A'] = (self.head_A(h[:, 0, 2]), tA, RN['a_w'])
        spk_ = self.text(TXT[ix]) + self.audio(AUD[ix]) * AFD[ix].unsqueeze(-1) + self.voice(VOI[ix]) + self.ctx_role[0]
        scn = self.scene(SCN[ix]) + self.ctx_role[1]
        ct = torch.cat([spk_ + self.clip_emb, scn + self.clip_emb], 1)         # [B, 6, d]
        aux['ctx'] = (self.head_ctx(ct.mean(1)), YB[ix], RN['aux_w'])
        toks = torch.cat([self.query.expand(B, -1, -1), ft, ct], 1)
        valid = torch.ones(toks.shape[:2], dtype=torch.bool, device=toks.device)
        self.last_drop_face = None
        if train:
            u = torch.rand(B, device=toks.device)
            drop_ctx = u < RN['p_drop_ctx']
            drop_face = (u >= RN['p_drop_ctx']) & (u < RN['p_drop_ctx'] + RN['p_drop_face'])
            valid[:, 1:10] &= ~drop_face.unsqueeze(1)
            valid[:, 10:] &= ~drop_ctx.unsqueeze(1)
            self.last_drop_face = drop_face
        out = self.enc(toks, src_key_padding_mask=~valid)
        return self.head(out[:, 0]), aux


def draw_cond(j):
    # R-mask: with prob. P_MASK pick uniformly one condition that changes this MCIS; none applicable -> unchanged
    el = ELIG_T[j]
    do = (torch.rand(len(j), device=DEVICE) < P_MASK) & el.any(1)
    w = el.float() + (~el.any(1, keepdim=True)).float()
    pick = torch.multinomial(w, 1).squeeze(1) + 1
    return torch.where(do, pick, torch.zeros_like(pick))


def predict_c(model, ix, k, bs=256):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(ix), bs):
            j = ix[i:i + bs]
            out.append(F.softmax(model(j, cond=torch.full((len(j),), k, dtype=torch.long, device=DEVICE))[0], -1).cpu())
    return torch.cat(out).numpy()


def nll_mean(p, y):
    return float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-7, None)).mean())


def train_eval_g28(arm, tr, dev, te, seed):
    seed_all(seed)
    hp = HP['role']
    model = RoleNetMask(FULL).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=hp['lr'], weight_decay=hp['wd'])
    y_dev = YB[dev].cpu().numpy()
    el_dev = ELIG[dev.cpu().numpy()]
    st = {'n': 0, 'masked': 0, 'C1': 0, 'C2': 0, 'C3': 0, 'masked_and_facedrop': 0}
    best, best_state, bad, best_ep = np.inf, None, 0, -1
    for ep in range(hp['epochs']):
        model.train()
        perm = tr[torch.randperm(len(tr), device=DEVICE)]
        for i in range(0, len(perm), hp['batch']):
            j = perm[i:i + hp['batch']]
            cond = draw_cond(j)                       # drawn in both arms so that their random streams stay aligned
            if arm == 'R-std':
                cond = torch.zeros_like(cond)
            logits, aux = model(j, train=True, cond=cond)
            st['n'] += len(j); st['masked'] += int((cond > 0).sum())
            for k in (1, 2, 3):
                st[f'C{k}'] += int((cond == k).sum())
            st['masked_and_facedrop'] += int(((cond > 0) & model.last_drop_face).sum())
            loss = F.cross_entropy(logits, YB[j])
            for l, t, w in aux.values():
                if (t >= 0).any():
                    loss = loss + w * F.cross_entropy(l, t, ignore_index=-100)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        terms = [nll_mean(predict_c(model, dev[torch.tensor(el_dev[:, k - 1], device=DEVICE)], k), y_dev[el_dev[:, k - 1]])
                 for k in (1, 2, 3) if el_dev[:, k - 1].any()]
        J = 0.5 * nll_mean(predict_c(model, dev, 0), y_dev) + 0.5 * (np.mean(terms) if terms else 0.0)
        if J < best:
            best, bad, best_ep = J, 0, ep
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= hp['patience']:
                break
    model.load_state_dict(best_state)
    probs = np.stack([predict_c(model, te, k) for k in range(4)])         # [4 conditions, n_te, 7]
    return probs, best, best_ep, st, int(el_dev.any(0).sum())


print("parameters:", f"{sum(p.numel() for p in RoleNetMask(FULL).parameters()) / 1e6:.3f}M")

## 5-fold episode cross-validation (same folds, early-stop episodes and seeds as G14)

In [ ]:
import re

OOF = {a: np.full((len(SEEDS), 4, N, 7), np.nan, np.float32) for a in ARMS}
log = []
t0 = time.time()
for f in range(N_OUTER):
    tr_eps = [e for e in EPS if FOLD[e] != f]
    dev_eps = sorted(random.Random(100 + f).sample(tr_eps, N_INNER_DEV))
    trr = np.where(np.isin(src, tr_eps))[0]
    fit_rows = np.where(np.isin(src, tr_eps) & ~np.isin(src, dev_eps))[0]
    dev_rows = np.where(np.isin(src, dev_eps))[0]
    te_rows = np.where(fold_of_row == f)[0]
    fit_clips = sorted(set(DEV.iloc[trr][['clip1', 'clip2', 'clip3']].values.ravel()))
    FACE, POOL, var = build_face_tensors(fit_clips)
    print(f"fold {f}: train {len(fit_rows)} | early-stop {len(dev_rows)} | eval {len(te_rows)} | dev MCIS eligible for "
          f"C1/C2/C3 {ELIG[dev_rows].sum(0).tolist()}", flush=True)
    tr, dev, te = T(fit_rows), T(dev_rows), T(te_rows)
    for arm in ARMS:
        for si, seed in enumerate(SEEDS):
            p, J, ep, st, n_terms = train_eval_g28(arm, tr, dev, te, seed + 1000 * f)
            OOF[arm][si][:, te_rows] = p
            yt = y_all[te_rows]
            log.append({'fold': f, 'arm': arm, 'seed': seed, 'J_dev': J, 'best_epoch': ep, 'dev_cond_terms': n_terms,
                        **{f'NLL_C{k}': nll_mean(p[k], yt) for k in range(4)}, **st})
            print(f"fold {f} {arm:<6} seed {seed:>3}: J_dev {J:.4f} (epoch {ep}) | NLL C0 {nll_mean(p[0], yt):.4f} | "
                  f"masked {st['masked'] / max(st['n'], 1) * 100:4.1f}% of training draws | "
                  f"{(time.time() - t0) / 60:.1f} min", flush=True)
            torch.cuda.empty_cache()

assert all(not np.isnan(v).any() for v in OOF.values())
LOG = pd.DataFrame(log)
LOG.to_csv(f"{OUT_DIR}/g28_fold_seed_log.csv", index=False)
np.savez(f"{OUT_DIR}/g28_oof_probs.npz", sample_id=DEV.sample_id.values, fold=fold_of_row, y=y_all, src=src,
         presence=PRES, eligible=ELIG, presence_oof_prior=PRES_OOF['prior'], presence_oof_flags=PRES_OOF['flags'],
         presence_oof_pairs=PRES_OOF['flags+pairs'], **{re.sub(r'[^0-9A-Za-z]+', '_', a): v for a, v in OOF.items()})
print("saved g28_oof_probs.npz (arm arrays: [seed, condition C0..C3, MCIS, 7]) and g28_fold_seed_log.csv")

## Masking actually applied during training

In [ ]:
m = LOG[LOG.arm == 'R-mask']
tot = m[['n', 'masked', 'C1', 'C2', 'C3', 'masked_and_facedrop']].sum()
print(f"R-mask: masked share of training draws {tot.masked / tot.n * 100:.1f}% (nominal {P_MASK * 100:.0f}%, lower because "
      f"some MCIS have no applicable condition)")
print("  condition frequency among masked draws:", {k: f"{tot[k] / max(tot.masked, 1) * 100:.1f}%" for k in ('C1', 'C2', 'C3')})
print(f"  masked draws whose whole face branch was also dropped by modality dropout: "
      f"{tot.masked_and_facedrop / max(tot.masked, 1) * 100:.1f}% (the L mask had no effect on the main head there)")
print("  R-std masked draws:", int(LOG[LOG.arm == 'R-std'].masked.sum()), "(expected 0)")
print("  best epoch per arm (mean):", LOG.groupby('arm').best_epoch.mean().round(1).to_dict())

## Results (fixed estimand: mean of per-seed NLL; two-level bootstrap with shared draws)

In [ ]:
S = len(SEEDS)
LOSS = {a: -np.log(np.clip(np.take_along_axis(OOF[a], y_all[None, None, :, None], -1)[..., 0], 1e-7, None))
        for a in ARMS}                                                   # [S, 4, N]
SETS = [np.ones(N, bool), ELIG[:, 0], ELIG[:, 1], ELIG[:, 2]]          # evaluation set of C0 (all), C1, C2, C3


def quantities(sidx, w):
    # every reported quantity for one draw: seeds sidx (indices into SEEDS), row weights w (episode multiplicities)
    M = {a: LOSS[a][sidx].mean(0) for a in ARMS}                        # mean over drawn seeds, [4, N]
    nll = lambda a, k, s: float((M[a][k] * w * s).sum() / max((w * s).sum(), 1e-12))
    q = {}
    for k in (1, 2, 3):
        s = SETS[k]
        q[f'drec_C{k}'] = nll('R-std', k, s) - nll('R-mask', k, s)
        q[f'sens_C{k}'] = nll('R-std', k, s) - nll('R-std', 0, s)
        q[f'sens_mask_C{k}'] = nll('R-mask', k, s) - nll('R-mask', 0, s)
        q[f'clean_on_C{k}'] = nll('R-mask', 0, s) - nll('R-std', 0, s)
        q[f'std_C{k}'], q[f'mask_C{k}'] = nll('R-std', k, s), nll('R-mask', k, s)
    q['drec'] = np.mean([q[f'drec_C{k}'] for k in (1, 2, 3)])
    q['dclean'] = nll('R-mask', 0, SETS[0]) - nll('R-std', 0, SETS[0])
    q['std_C0'], q['mask_C0'] = nll('R-std', 0, SETS[0]), nll('R-mask', 0, SETS[0])
    return q


point = quantities(np.arange(S), np.ones(N))
rng = np.random.default_rng(0)
gidx = [np.where(src == e)[0] for e in np.unique(src)]
draws = []
for _ in range(N_BOOT):
    w = np.zeros(N)
    for i in rng.integers(0, len(gidx), len(gidx)):
        w[gidx[i]] += 1
    draws.append(quantities(rng.integers(0, S, S), w))
D = pd.DataFrame(draws)
CI = {k: np.percentile(D[k], [2.5, 97.5]) for k in D}


def show(k, label):
    lo, hi = CI[k]
    print(f"  {label:<58} {point[k]:+.4f} [{lo:+.4f}, {hi:+.4f}]")


print(f"NLL (mean of per-seed NLL), C0 all MCIS: R-std {point['std_C0']:.4f} | R-mask {point['mask_C0']:.4f}")
for k in (1, 2, 3):
    print(f"  {COND[k - 1]:<32} n={SETS[k].sum():4d}: R-std {point[f'std_C{k}']:.4f} | R-mask {point[f'mask_C{k}']:.4f}")
print("\n== primary ==")
show('drec', "Δ_rec = mean_k,s [NLL_std − NLL_mask] under C1–C3")
print("\n== secondary ==")
show('dclean', "Δ_clean = NLL_mask − NLL_std on C0, all MCIS (> 0 = cost)")
for k in (1, 2, 3):
    show(f'drec_C{k}', f"Δ_rec,{k} ({COND[k - 1]})")
for k in (1, 2, 3):
    show(f'sens_C{k}', f"sensitivity R-std, C{k} − C0 on the C{k} set")
    show(f'sens_mask_C{k}', f"sensitivity R-mask, C{k} − C0 on the C{k} set")
    show(f'clean_on_C{k}', f"NLL_mask − NLL_std under C0 on the C{k} set")
print("\n== recovery share (only where the sensitivity CI lower bound > 0) ==")
for k in (1, 2, 3):
    if CI[f'sens_C{k}'][0] > 0:
        r = D[f'drec_C{k}'] / D[f'sens_C{k}']
        lo, hi = np.percentile(r, [2.5, 97.5])
        print(f"  C{k}: {point[f'drec_C{k}'] / point[f'sens_C{k}']:.2f} [{lo:.2f}, {hi:.2f}]  (not a share of recovered information)")
    else:
        print(f"  C{k}: not reported (sensitivity CI includes 0 or is negative); see the two NLL differences above")

lo, hi = CI['drec']
verdict = ('MASKED TRAINING HELPS ADAPTATION to the interventions tried' if lo > 0 else
           'MASKED TRAINING MAKES THE FORECAST UNDER MASKING WORSE' if hi < 0 else
           'NOT SHOWN that this policy helps; cannot tell lost evidence from lack of adaptation')
print(f"\n== G28 reading (fixed rule): {verdict} ==")
if lo > 0 and CI['dclean'][0] > 0:
    print("   with a cost on the data as they are (Δ_clean CI above 0): a trade-off, not robustness without cost")

print("\n== descriptive: UAR (per-seed mean / 10-seed ensemble) and ensemble NLL ==")
rows = []
for a in ARMS:
    for k in range(4):
        s = SETS[k]
        per = [war_uar(OOF[a][si, k, s].argmax(1), y_all[s], 7)[1] for si in range(S)]
        ens = OOF[a][:, k, s].mean(0)
        rows.append({'arm': a, 'condition': ['C0', 'C1', 'C2', 'C3'][k],
                     'n': int(s.sum()), 'UAR_seed_mean': np.mean(per), 'UAR_ensemble': war_uar(ens.argmax(1), y_all[s], 7)[1],
                     'NLL_seed_mean': LOSS[a][:, k, s].mean(), 'NLL_ensemble': nll_mean(ens, y_all[s])})
SUM = pd.DataFrame(rows)
print(SUM.round(4).to_string(index=False))
SUM.to_csv(f"{OUT_DIR}/g28_summary.csv", index=False)
pd.DataFrame({k: [point[k], CI[k][0], CI[k][1]] for k in point}, index=['point', 'lo', 'hi']).T.to_csv(
    f"{OUT_DIR}/g28_estimands.csv")
print("saved g28_summary.csv and g28_estimands.csv")